In [ ]:
# Import necessary libraries
import os
from ultralytics import YOLO
import numpy as np
import cv2
# Load the trained model
MODEL_PATH = os.path.expanduser('~/TrafficRouting/detection/models/yolo11x_visdrone_50epoch.pt')
model = YOLO(MODEL_PATH)

In [ ]:
# Traffic weights
WEIGHTS = {
    "car": 1.0,
    "van": 1.5,
    "truck": 3.0,
    "bus": 3.0,
    "motor": 0.5,
    "bicycle": 0.3
}

In [ ]:
# Function to analyze traffic in an image
def analyze_image(image_path):
    results = model.predict(source=image_path, conf=0.25, verbose=False)[0]

    traffic_counts = {k: 0 for k in WEIGHTS.keys()}
    boxes = results.boxes

    # Count objects
    for box in boxes:
        cls = int(box.cls[0].item())
        label = model.names[cls]

        if label in traffic_counts:
            traffic_counts[label] += 1

    # Compute traffic score
    traffic_score = sum(
        traffic_counts[k] * WEIGHTS[k] for k in traffic_counts
    )

    # Classify congestion
    if traffic_score < 10:
        level = "LOW"
    elif traffic_score < 25:
        level = "MEDIUM"
    else:
        level = "HIGH"

    return traffic_counts, traffic_score, level

In [ ]:
# Function to decide routing based on traffic analysis
def route_decision(image_path):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    results = model.predict(source=image_path, conf=0.25, verbose=False)[0]

    zones = {
        "left": 0,
        "center": 0,
        "right": 0
    }

    for box in results.boxes:
        x1, y1, x2, y2 = [v.item() for v in box.xyxy[0]]
        cls = int(box.cls[0].item())
        label = model.names[cls]

        if label not in WEIGHTS:
            continue

        center_x = (x1 + x2) / 2

        if center_x < w / 3:
            zone = "left"
        elif center_x < 2 * w / 3:
            zone = "center"
        else:
            zone = "right"

        zones[zone] += WEIGHTS[label]

    best_route = min(zones, key=zones.get)

    return zones, best_route

In [ ]:
# Main function
if __name__ == "__main__":
    IMAGE_PATH = os.path.expanduser('~/TrafficRouting/detection/images/img0.jpg')

    counts, score, level = analyze_image(IMAGE_PATH)
    zones, route = route_decision(IMAGE_PATH)

    print("\n=== TRAFFIC ANALYSIS ===")
    print("Counts:", counts)
    print(f"Score: {score:.2f}")
    print("Congestion Level:", level)

    print("\n=== ROUTING ===")
    print("Zone Scores:", zones)
    print("Recommended Route:", route.upper())

In [ ]:
# Process all images from VisDrone dataset and build graph data
import re
from collections import defaultdict
from sklearn.cluster import KMeans
import networkx as nx
import pandas as pd
from pathlib import Path

# Paths to VisDrone images
VISDRONE_IMAGE_DIR = Path('/Users/aaliyahcreech/AdvAI-Final-Project/AdvAI_Final/datasets/VisDrone/images')

def parse_filename(filename):
    """Parse VisDrone filename format"""
    # Extract sequence number and frame number
    match = re.match(r'(\d+)_(\d+)_d_\d+\.jpg', filename)
    if match:
        sequence_id = int(match.group(1))
        frame_num = int(match.group(2))
        return sequence_id, frame_num
    return None, None

def get_all_visdrone_images():
    """Get all image paths from train/val/test splits"""
    image_paths = []
    for split in ['train', 'val', 'test']:
        split_dir = VISDRONE_IMAGE_DIR / split
        if split_dir.exists():
            for img_path in split_dir.glob('*.jpg'):
                image_paths.append(img_path)
    return image_paths

def process_all_images():
    """Process all images and store results keyed by sequence and frame"""
    image_paths = get_all_visdrone_images()
    print(f"Found {len(image_paths)} images")
    
    # Dictionary: {sequence_id: {frame_num: {'image_path': ..., 'traffic_score': ..., 'vehicles': [...]}}}
    sequence_data = defaultdict(dict)
    
    for i, img_path in enumerate(image_paths):
        seq_id, frame_num = parse_filename(img_path.name)
        if seq_id is None:
            continue
            
        # Run YOLO detection
        results = model.predict(source=str(img_path), conf=0.25, verbose=False)[0]
        
        vehicles = []
        traffic_score = 0
        
        for box in results.boxes:
            cls = int(box.cls[0].item())
            label = model.names[cls]
            
            if label in WEIGHTS:
                x1, y1, x2, y2 = [v.item() for v in box.xyxy[0]]
                center_x = (x1 + x2) / 2
                center_y = (y1 + y2) / 2
                vehicles.append({
                    'label': label,
                    'center': (center_x, center_y),
                    'weight': WEIGHTS[label]
                })
                traffic_score += WEIGHTS[label]
        
        sequence_data[seq_id][frame_num] = {
            'image_path': str(img_path),
            'traffic_score': traffic_score,
            'vehicles': vehicles,
            'num_vehicles': len(vehicles)
        }
        
        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1}/{len(image_paths)} images...")
    
    print(f"Processed {len(sequence_data)} sequences")
    return dict(sequence_data)

# Process all images
print("Processing all VisDrone images...")
sequence_data = process_all_images()

In [ ]:
# Build road graph using NetworkX with K-means clustering
def build_road_graph(sequence_data, n_clusters=6):
    """
    Build a graph for each sequence where:
    - Nodes: clusters of vehicle centers (representing road regions)
    - Edges: connectivity between regions based on spatial proximity
    """
    graphs = {}
    
    for seq_id, frames in sequence_data.items():
        if len(frames) < 2:
            continue
            
        # Collect all vehicle centers from all frames in this sequence
        all_centers = []
        frame_features = []
        
        for frame_num, frame_data in sorted(frames.items()):
            vehicles = frame_data['vehicles']
            if vehicles:
                for v in vehicles:
                    all_centers.append(v['center'])
            
            # Feature vector for this frame: [traffic_score, num_vehicles]
            frame_features.append([frame_data['traffic_score'], frame_data['num_vehicles']])
        
        if len(all_centers) < n_clusters:
            continue
        
        # K-means clustering on vehicle centers
        centers_array = np.array(all_centers)
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(centers_array)
        cluster_centers = kmeans.cluster_centers_
        
        # Build NetworkX graph
        G = nx.Graph()
        
        # Add nodes (road regions)
        for i, center in enumerate(cluster_centers):
            G.add_node(i, pos=center, cluster_size=np.sum(cluster_labels == i))
        
        # Add edges between clusters based on proximity
        for i in range(n_clusters):
            for j in range(i + 1, n_clusters):
                dist = np.linalg.norm(cluster_centers[i] - cluster_centers[j])
                # Add edge if clusters are close enough (within 200 pixels)
                if dist < 200:
                    G.add_edge(i, j, weight=1/dist)  # Inverse distance as weight
        
        # Add frame-level features as node attributes
        for frame_num, frame_data in frames.items():
            for node in G.nodes():
                G.nodes[node]['frame_features'] = frame_features
        
        graphs[seq_id] = {
            'graph': G,
            'cluster_centers': cluster_centers,
            'total_frames': len(frames),
            'total_vehicles': len(all_centers)
        }
    
    print(f"Built graphs for {len(graphs)} sequences")
    return graphs

# Build graphs for all sequences
print("Building road graphs...")
road_graphs = build_road_graph(sequence_data, n_clusters=6)